[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_strain_vmap.ipynb)

# Batched RVE Solves via `jax.vmap`

A companion to [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb): the same two-phase
composite RVE and elastic solver (`problems.mechanics.solve_mechanics`), but solving **multiple
macroscopic load cases in one call** via `jax.vmap` instead of a Python loop.

`solve_mechanics` was written as an ordinary single-example function — it has no batch dimension
anywhere in its implementation. `jax.vmap` adds one automatically: it traces the function once,
inserts JAX's batching rules for every primitive op inside it (FFT, `einsum`, the CG solve's
`lax.while_loop`, ...), and returns a new function that accepts a stacked batch of `eps_bar` and
returns the same `list[IncrementResult]` structure `solve_mechanics` always returns (one element,
for the default `stepping="single"`), with every array leaf inside it — `.solution.sigma`,
`.solution.converged`, ... — carrying the extra batch axis. No changes to `solve_mechanics` itself.

Only `eps_bar` is batched here; `n`, `L`, `phase`, `materials` are shared across the whole batch
(same RVE, same materials, different loading).

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

Same square-packed glass-fiber/epoxy RVE as `lin-elastic_strain.ipynb` — see that notebook for the
full walkthrough of `generation.rve.make_square_composite_rve`.

In [ ]:
from generation.rve import make_square_composite_rve

phi     = 0.5      # target fiber volume fraction
r_fiber = 0.005     # fiber radius [mm]
dx      = 0.0002    # target voxel size [mm]

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi,
    r_fiber=r_fiber,
    dx=dx,
    N_min=32,
    nz=1,
)

print("grid n :", n)
print("total voxels Nv:", int(np.prod(n)))
print("domain L [mm]:", tuple(f"{float(Li):.5g}" for Li in L))
print("fiber volume fraction (actual):", f"{phi_act:.4f}")

### Quick 3-D look (PyVista)

An interactive voxel render of the same phase field (drag to rotate, scroll to zoom) -- `n[2] == 1`
here, so it renders as a thin slab, but the snippet is unchanged on a thicker RVE. `html` is
PyVista's self-contained Jupyter backend: the whole scene is embedded inline, so it works the same
in Colab or a live local kernel, with no server process to keep alive.

In [ ]:
import pyvista as pv
pv.set_jupyter_backend("html")   # self-contained interactive widget (drag to rotate, scroll to zoom) -- inlined, no live server needed

spacing = tuple(Li / ni for Li, ni in zip(L, n))
grid = pv.ImageData(dimensions=(n[0] + 1, n[1] + 1, n[2] + 1), spacing=spacing)
grid.cell_data["phase"] = np.asarray(phase_np).reshape(n).flatten(order="F")

pl = pv.Plotter(window_size=(500, 400))
pl.add_mesh(grid, scalars="phase", cmap="viridis", show_scalar_bar=False)
pl.camera_position = "iso"
pl.show()

## Materials

In [ ]:
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from materialmodels.assembly import describe_materials

matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")
materials = [matrix, fiber]   # index 0 = matrix, 1 = fibre -- matches the phase labels below

phase = jnp.array(phase_np.reshape(-1))   # (nx,ny,nz) -> (Nv,)

describe_materials(materials)

## Batched macroscopic strains

We solve three canonical load cases at once: in-plane shear, and uniaxial extension along X and
Y. Stacking them along a new leading axis is the only "batching" needed — `jax.vmap` handles the
rest.

In [ ]:
eps_shear = jnp.array([
    [0.0,    1.0e-3, 0.0],
    [1.0e-3, 0.0,    0.0],
    [0.0,    0.0,    0.0],
])
eps_uniax_x = jnp.array([
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
    [0.0,    0.0, 0.0],
])
eps_uniax_y = jnp.array([
    [0.0, 0.0,    0.0],
    [0.0, 1.0e-3, 0.0],
    [0.0, 0.0,    0.0],
])

eps_bar_batch = jnp.stack([eps_shear, eps_uniax_x, eps_uniax_y])
load_case_labels = ["xy shear", "uniaxial x", "uniaxial y"]

print("eps_bar_batch shape:", eps_bar_batch.shape)   # (B, 3, 3), B = 3 load cases

## Solve — `jax.vmap(solve_mechanics)`

`solve_mechanics` takes several non-batched arguments (`n`, `L`, `phase`, `materials`) ahead of
`eps_bar`; wrapping it in a one-argument closure keeps `jax.vmap`'s default `in_axes=0` (batch the
first, and only, argument) instead of having to spell out an `in_axes` tuple by position.

`jax.jit(jax.vmap(...))` additionally compiles the whole batch as one XLA program — still one
traced function, so it costs one compilation regardless of batch size, and the same composition
works with `jax.grad` too since nothing here is vmap- or batch-specific code.

`solve_mechanics` always returns `list[IncrementResult]`; under vmap the list itself stays length
1 (it's the default `stepping="single"`), only the array leaves inside `.solution` pick up the
batch axis — so `solve_batch(eps_bar_batch)[0].solution` is the batched `ElasticitySolution`.

In [ ]:
from problems.mechanics import solve_mechanics

def solve_one(eps_bar):
    return solve_mechanics(
        n, L, phase, materials, eps_bar,
        scheme="rotated", toler_lin=1e-6, maxiter=1000,
    )

solve_batch = jax.jit(jax.vmap(solve_one))

sol_batch = solve_batch(eps_bar_batch)[0].solution
eps_batch, sigma_batch, delta_batch, converged_batch = (
    sol_batch.eps, sol_batch.sigma, sol_batch.delta, sol_batch.converged
)

print("eps_batch shape   :", eps_batch.shape)     # (B, 3, 3, Nv)
print("sigma_batch shape :", sigma_batch.shape)   # (B, 3, 3, Nv)
for label, conv in zip(load_case_labels, converged_batch):
    print(f"  {label:12s} converged: {bool(conv)}")

## Per-case average stress

In [ ]:
print(f"{'load case':12s}  {'sigma_xx':>10s}  {'sigma_yy':>10s}  {'tau_xy':>10s}")
for i, label in enumerate(load_case_labels):
    sxx = float(jnp.mean(sigma_batch[i, 0, 0]))
    syy = float(jnp.mean(sigma_batch[i, 1, 1]))
    sxy = float(jnp.mean(sigma_batch[i, 1, 0]))
    print(f"{label:12s}  {sxx:10.3f}  {syy:10.3f}  {sxy:10.3f}")

## Sanity check — vmap matches a sequential loop

`jax.vmap` doesn't change *what* gets computed, only how it's expressed and executed — each batch
element is solved exactly as if `solve_mechanics` had been called on it alone. Confirming that
here, once, is cheap insurance against `vmap`'s batching rules doing something unexpected to the
CG solve's `lax.while_loop` internals.

In [ ]:
sigma_ref = jnp.stack([solve_one(eps_bar_batch[i])[0].solution.sigma for i in range(len(load_case_labels))])
matches = jnp.allclose(sigma_batch, sigma_ref, atol=1e-8)
print("vmap matches sequential solve_mechanics calls:", bool(matches))
assert matches

## Visualize the von Mises stress field per load case

In [ ]:
from post.fields import field_to_grid, von_mises

extent = [0.0, L[0], 0.0, L[1]]
fig, axes = plt.subplots(1, len(load_case_labels), figsize=(4 * len(load_case_labels), 4))

for i, (ax, label) in enumerate(zip(axes, load_case_labels)):
    sigma_grid = field_to_grid(sigma_batch[i], n)
    vm_grid = von_mises(sigma_grid)
    im = ax.imshow(vm_grid[:, :, 0].T, origin="lower", cmap="inferno", extent=extent)
    ax.set_title(f"von Mises — {label}")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Export each load case to XDMF/HDF5

The batch dimension is only a `jax.vmap` implementation detail -- for output each load case is
its own independent field, so each gets its own `IncrementalWriter` file (one increment each)
rather than one file with a spurious "time" axis over unrelated load cases.

In [ ]:
import os

from post.fields import compute_displacement, to_voigt
from utils.io.xdmf_writer import IncrementalWriter

output_dir = "output" if IN_COLAB else "../output/notebooks/lin-elastic_vmap"
os.makedirs(output_dir, exist_ok=True)

for i, label in enumerate(load_case_labels):
    stem = label.replace(" ", "_")
    eps_i, sigma_i = eps_batch[i], sigma_batch[i]
    eps_grid   = field_to_grid(eps_i, n)
    sigma_grid = field_to_grid(sigma_i, n)
    u_grid     = compute_displacement(eps_i, eps_bar_batch[i], n, L)
    vm_grid    = von_mises(sigma_grid)

    with IncrementalWriter(f"{output_dir}/{stem}", grid_shape=n, grid_length=L) as writer:
        writer.write_increment(0, {
            "phase":        phase_np.astype(np.float64),
            "displacement": u_grid.astype(np.float64),
            "strain":       to_voigt(eps_grid).astype(np.float64),
            "stress":       to_voigt(sigma_grid).astype(np.float64),
            "von_mises":    vm_grid.astype(np.float64),
        }, time=0.0)
    print(f"Wrote {output_dir}/{stem}.h5 / .xdmf")

## Next steps

- Batching over `eps_bar` is one axis choice; the same pattern applies to batching over
  microstructures (different `phase` fields) or materials, as long as every batch element shares
  the same grid shape `n` — `vmap` needs uniformly-shaped arrays.
- `jax.jit(jax.vmap(...))` compiles once per batch size; changing the batch size triggers a
  recompile, same as any other JIT-traced shape change.
- Since `solve_mechanics` is an ordinary JAX function with no batch-specific code,
  `jax.vmap(jax.grad(...))` composes the same way — differentiating through a batch of solves
  needs no special-casing either.
- Open any of `output/notebooks/lin-elastic_vmap/*.xdmf` in ParaView (`Xdmf3ReaderT`) to inspect a
  load case's full field data directly, beyond the von Mises snapshot plotted above.